# 言語資源のクレンジングと重複排除（データリークの実験）

書籍 第5章「言語資源のつくり方」（5-1 言語資源の入手方法、5-4 コーパス作成）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、コーパス構築時に不可欠な前処理と品質チェックを体験します。

1. **テキストのクレンジングと正規化**: HTML タグ除去、Unicode 正規化（NFKC）、余分な空白の統一
2. **完全重複・近似重複（ニアデュープリケート）の検出**: 文字 n-gram の Jaccard 係数による類似文書の特定
3. **データリーク実験**: 重複を除去せずに訓練/テスト分割すると評価スコアがどれだけ過大評価されるかを確認

## 1. テキストのクレンジングと Unicode 正規化（5-4）

Web クロールなどで集めたテキストには、HTML タグ、全角英数字・記号の表記揺れ（`ＡＩ` vs `AI`、`ｶﾀｶﾅ` vs `カタカナ`）、連続空白などが混在します。

In [1]:
import html
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)                 # HTMLタグ除去
    text = re.sub(r"https?://\S+", " ", text)            # URL除去
    text = unicodedata.normalize("NFKC", text)           # Unicode正規化（全角英数→半角、半角カナ→全角等）
    text = re.sub(r"\s+", " ", text).strip()             # 空白の統一
    return text

dirty_samples = [
    "<p>ＮＬＰの教科書（２０２４年）を読みました！</p> https://example.com",
    "ﾃｽﾄ環境で　　BERT＆GPTを動かした。",
]
for s in dirty_samples:
    print("前:", s)
    print("後:", clean_text(s), "\n")

前: <p>ＮＬＰの教科書（２０２４年）を読みました！</p> https://example.com
後: NLPの教科書(2024年)を読みました! 

前: ﾃｽﾄ環境で　　BERT＆GPTを動かした。
後: テスト環境で BERT&GPTを動かした。 



---

## 2. 文字 n-gram の Jaccard 係数による近似重複検出

完全一致だけでなく、文末の記号や 1〜2 語だけが異なる**近似重複（ニアデュープリケート）**を検出するため、文字 3-gram 集合の **Jaccard 係数**（$|A \cap B| / |A \cup B|$）を計算します。

In [2]:
def char_trigrams(text):
    t = clean_text(text)
    return set(t[i:i + 3] for i in range(max(1, len(t) - 2)))

def jaccard(a, b):
    sa, sb = char_trigrams(a), char_trigrams(b)
    return len(sa & sb) / len(sa | sb)

s1 = "駅前で美味しいラーメンを食べました。"
s2 = "駅前で美味しいラーメンを食べました！"      # 記号違い（近似重複）
s3 = "駅前のレストランでパスタを食べました。"      # 別文書

print(f"s1 vs s2 (近似重複): Jaccard = {jaccard(s1, s2):.3f}")
print(f"s1 vs s3 (別文書)  : Jaccard = {jaccard(s1, s3):.3f}")

s1 vs s2 (近似重複): Jaccard = 0.882
s1 vs s3 (別文書)  : Jaccard = 0.179


---

## 3. 重複排除を怠ったときのデータリーク実験

Web から集めたレビューデータに「同じ内容のテンプレート投稿やコピペ（近似重複）」が多く含まれている状況を模擬します。
**重複を除去せずにランダム分割**すると、訓練データとほぼ同じ文がテストデータにも紛れ込み（データリーク）、テスト精度が実際より高く見えてしまいます。

In [3]:
rng = np.random.default_rng(42)

# それぞれ表現が異なるユニークな文（pos 20件, neg 20件）を作成
pos_topics = [
    "部屋の清掃が隅々まで行き届いていて気持ちよく過ごせました",
    "温泉の大浴場が広くて旅の疲れがしっかり取れました",
    "朝食の焼きたてパンと地元野菜のサラダがとても美味しかったです",
    "スタッフの笑顔と丁寧な案内のおかげで安心して滞在できました",
    "駅から歩いてすぐの立地で大きな荷物があっても便利でした",
    "ベッドの寝心地が良く朝までぐっすり眠ることができました",
    "静かな環境でオンライン会議や仕事にも集中できました",
    "チェックイン前の荷物預かりに快く対応してもらい助かりました",
    "窓からの夜景が美しく記念日の良い思い出になりました",
    "アメニティが充実していて手ぶらで泊まれて快適でした",
    "館内のWi-Fiが高速で動画視聴もストレスがありませんでした",
    "貸切風呂の予約が取りやすく家族でゆっくり入浴できました",
    "夕食の郷土料理がボリューム満点で大変満足しました",
    "駐車場が無料で広いため車での旅行でも安心でした",
    "空気清浄機や加湿器が部屋に完備されていて快適でした",
    "ラウンジの無料コーヒーサービスが嬉しかったです",
    "防音がしっかりしており廊下の音も気になりませんでした",
    "子供の宿泊向けサービスが丁寧で家族連れに優しい宿でした",
    "価格以上の設備とサービスでコストパフォーマンスが高いです",
    "また機会があればぜひリピートしたいと思えるホテルでした",
]
neg_topics = [
    "部屋の隅にホコリが残っていて清掃状態に不満がありました",
    "大浴場の脱衣所が狭く混雑していて落ち着きませんでした",
    "朝食の品数が少なく補充も遅かったのが残念でした",
    "フロントの対応が事務的で質問しても素っ気ない印象でした",
    "駅から遠く坂道も多いため徒歩での移動が大変でした",
    "マットレスが硬すぎて夜中に何度も目が覚めてしまいました",
    "外の車の騒音が夜遅くまで響いてよく眠れませんでした",
    "チェックインの手続きに30分以上待たされて疲れました",
    "建物の古さが目立ち水回りの設備に使いづらさを感じました",
    "部屋の照明が暗くデスクで作業するには不便でした",
    "Wi-Fiの接続が頻繁に切れて仕事になりませんでした",
    "エアコンの温度調節が効かず部屋が暑いままでした",
    "壁が薄く隣の部屋の話し声が丸聞こえで気になりました",
    "エレベーターが1基しかなく朝の移動に時間がかかりました",
    "シャワーの水圧が弱くお湯の温度も安定しませんでした",
    "駐車場の案内が分かりにくく周辺を何度も回りました",
    "エレベーター前の自動販売機が故障したままでした",
    "予約時のリクエストが伝わっておらず確認に時間がかかりました",
    "内容を考えると宿泊料金がやや割高に感じました",
    "改善されない限り次回の利用は控えようと思います",
]

unique_texts = pos_topics + neg_topics
unique_labels = [1] * len(pos_topics) + [0] * len(neg_topics)

# コーパスに「近似重複（末尾の記号違いや再投稿）」が混入している状況を模擬
leaked_texts = []
leaked_labels = []
for idx, (t, y) in enumerate(zip(unique_texts, unique_labels)):
    leaked_texts.append(t)
    leaked_labels.append(y)
    if idx % 2 == 0:  # 半分の文がテンプレート的に 5 倍に重複しているとする
        for suffix in ["。", "！", "♪", "（再投稿）"]:
            leaked_texts.append(t + suffix)
            leaked_labels.append(y)

print(f"重複ありコーパス: {len(leaked_texts)} 件")

# A) 重複を除去せずそのまま分割（データリークあり）
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(
    leaked_texts, leaked_labels, test_size=0.25, stratify=leaked_labels, random_state=42
)

# B) Jaccard >= 0.75 を近似重複とみなして重複排除してから分割（クリーン）
dedup_texts, dedup_labels = [], []
for t, y in zip(leaked_texts, leaked_labels):
    if not any(jaccard(t, kept) >= 0.75 for kept in dedup_texts):
        dedup_texts.append(t)
        dedup_labels.append(y)

print(f"重複排除後コーパス: {len(dedup_texts)} 件")
X_tr_clean, X_te_clean, y_tr_clean, y_te_clean = train_test_split(
    dedup_texts, dedup_labels, test_size=0.25, stratify=dedup_labels, random_state=42
)

def eval_clf(X_tr, y_tr, X_te, y_te):
    clf = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3)), LogisticRegression(max_iter=1000))
    clf.fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

acc_leak = eval_clf(X_tr_leak, y_tr_leak, X_te_leak, y_te_leak)
acc_clean = eval_clf(X_tr_clean, y_tr_clean, X_te_clean, y_te_clean)

pd.DataFrame([
    {"実験条件": "A. 重複排除なし（データリークあり）", "データ件数": len(leaked_texts), "テスト正解率": round(acc_leak, 3)},
    {"実験条件": "B. 近似重複排除後（真の汎化性能）", "データ件数": len(dedup_texts), "テスト正解率": round(acc_clean, 3)},
])

重複ありコーパス: 120 件
重複排除後コーパス: 40 件


,実験条件,データ件数,テスト正解率
0,A. 重複排除なし（データリークあり）,120,0.9
1,B. 近似重複排除後（真の汎化性能）,40,0.5


重複を除去しないまま分割すると（条件 A）、訓練データに見たことがあるのとほぼ同じ文がテストデータに混ざるため、見かけの正解率が過大評価されてしまいます。実運用や論文評価では、必ず**データ分割の前に完全重複・近似重複を排除する**ことが重要です。